# Module 2 · Alignment
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 1–2 · Days 6–10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M2_Alignment.ipynb)

**By the end of this module:** you can prepare a reference, map reads with BWA-MEM2, mark duplicates with Picard, understand base-quality recalibration and judge coverage.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

### Catch up on earlier modules
This rebuilds the outputs of Modules 1–1 (about 2–4 minutes). Skip it if you ran them in this same session.

In [ ]:
!bash setup/catchup.sh 1

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

## Day 6 · Reference genome preparation
Aligners and callers need **index files** to look up positions quickly: `.fai` (samtools), `.dict` (GATK/Picard) and the BWA-MEM2 index.

⚠️ This training reference is chromosome 20 of hg19/GRCh37 with most bases outside the workshop region masked as `N`. This keeps the lab small, but it means some reads look uniquely mapped here when, in the full genome, they could map elsewhere.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
samtools faidx run/input/ref.fa
samtools dict run/input/ref.fa -o run/input/ref.dict
bwa-mem2 index run/input/ref.fa 2>/dev/null
cat run/input/ref.fa.fai
ls -lh run/input/ref.fa*

> ✍️ **Your turn**
>
> 1. How long is the reference chromosome (from `.fai`)?
> 2. Why must the reference used for alignment be exactly the same one used for variant calling?

## Day 7 · Mapping reads with BWA-MEM2
BWA-MEM2 finds where each read pair best fits the reference. The **read group** (`-R`) labels the sample; GATK refuses BAMs without one.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
mkdir -p run/align
bwa-mem2 mem -t 2 -R '@RG\tID:HG001.region\tSM:HG001\tPL:ILLUMINA\tLB:lib1' run/input/ref.fa \
    run/qc/R1.fq.gz run/qc/R2.fq.gz 2>/dev/null | samtools sort -o run/align/sorted.bam -
samtools index run/align/sorted.bam
samtools flagstat run/align/sorted.bam

> ✍️ **Your turn**
>
> 1. What percentage of reads mapped? What percentage are properly paired?
> 2. Why do we sort the BAM by coordinate?

## Day 8 · Post-alignment processing: duplicates and recalibration
**PCR duplicates** are copies of the same DNA fragment. Counting them as independent evidence can make a sequencing error look like a real variant, so we mark them with Picard.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
picard MarkDuplicates -I run/align/sorted.bam -O run/align/HG001.bam \
    -M run/align/duplicates.txt --VERBOSITY ERROR 2>/dev/null
samtools index run/align/HG001.bam
grep -A 2 "^LIBRARY" run/align/duplicates.txt | cut -f 1-9

### Base Quality Score Recalibration (BQSR)
Sequencers report a quality for each base, but those scores can be systematically wrong (for example, by machine cycle). GATK **BaseRecalibrator** learns the errors at sites that are *not* known variants, then **ApplyBQSR** corrects the scores.

BQSR needs a large training set and a **known-sites** file (for example dbSNP). A 10 kb region is far too small to train on, so here you study the commands and plan the experiment. On a full genome you would run:
```bash
gatk BaseRecalibrator -R ref.fa -I sample.bam --known-sites dbsnp.vcf.gz -O recal.table
gatk ApplyBQSR -R ref.fa -I sample.bam --bqsr-recal-file recal.table -O sample.recal.bam
gatk AnalyzeCovariates -bqsr recal.table -plots recal.pdf
```

> ✍️ **Your turn**
>
> 1. How many duplicate reads did Picard mark, and what is the duplication rate?
> 2. Why must known variant sites be excluded when learning base-quality errors?
> 3. Write a before/after BQSR plan: what you would keep constant, one covariate to inspect, and the change you would expect.

## Day 9 · Coverage and quality metrics

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
samtools depth -aa -r chr20:10000001-10010000 run/align/HG001.bam > run/align/depth.tsv
awk '{s+=$3; if($3>=10) n++} END {printf "mean depth %.1fx, %d of %d bases at >=10x\n", s/NR, n, NR}' run/align/depth.tsv

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
d = pd.read_csv("run/align/depth.tsv", sep="\t", names=["chrom", "pos", "depth"])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(d.pos, d.depth, lw=0.6); ax[0].set(title="Depth along the target (HG001, chr20, GRCh37)", xlabel="position", ylabel="read depth")
ax[1].hist(d.depth.clip(upper=100), bins=20); ax[1].set(title="Distribution of depth", xlabel="read depth (100 = 100 or more)", ylabel="bases")
plt.tight_layout(); plt.show()

> ✍️ **Your turn**
>
> 1. Report the mean depth and the % of target bases at ≥10×, ≥20× and ≥40× (edit the `awk` command).
> 2. A single position can have depth 60 while the region has 99% of bases at ≥10×. Explain why these answer different questions.

## Day 10 · Alignment QC decision
**Module 2 deliverable:** a short table with mapping rate, properly-paired rate, duplicate rate, mean depth and % bases ≥10×, plus one limitation of this masked reference. Decide whether the BAM is fit for variant calling.